# SI10-2026 | Ponderada | Análise de Sensibilidade em Métricas de Interface Digital

Nesta atividade, você vai analisar quais variáveis de uma interface digital têm maior impacto sobre a taxa de conversão.

A entrega deve ser feita neste notebook, com código, tabelas, gráficos e respostas curtas.

## Contexto

Uma equipe de produto quer decidir qual métrica de interface deve receber prioridade no próximo ciclo de melhoria.

Os dados representam observações diárias de um aplicativo de compras.

A métrica alvo é a taxa de conversão.

As variáveis de entrada são taxa de abandono do carrinho, profundidade média de scroll e tempo até o primeiro clique em produto.

## Preparação

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.precision", 3)

## Dados

Execute a célula abaixo para criar a base da atividade.

In [2]:
rng = np.random.default_rng(42)
n_dias = 180

taxa_abandono = rng.normal(48, 8, n_dias).clip(25, 75)
profundidade_scroll = rng.normal(62, 12, n_dias).clip(25, 95)
tempo_primeiro_clique = rng.normal(7, 2.2, n_dias).clip(2, 15)

ruido = rng.normal(0, 0.35, n_dias)
taxa_conversao = (
    7.5
    - 0.055 * taxa_abandono
    + 0.026 * profundidade_scroll
    - 0.085 * tempo_primeiro_clique
    + ruido
).clip(0.5, 9.0)

df = pd.DataFrame({
    "data": pd.date_range("2026-01-01", periods=n_dias, freq="D"),
    "taxa_abandono_carrinho_pct": taxa_abandono,
    "profundidade_scroll_pct": profundidade_scroll,
    "tempo_primeiro_clique_s": tempo_primeiro_clique,
    "taxa_conversao_pct": taxa_conversao,
})

df.head()

,data,taxa_abandono_carrinho_pct,profundidade_scroll_pct,tempo_primeiro_clique_s,taxa_conversao_pct
0,2026-01-01,50.438,77.672,6.664,5.589
1,2026-01-02,39.680,64.633,7.843,6.042
2,2026-01-03,54.004,57.069,9.200,5.318
3,2026-01-04,55.525,75.275,4.671,5.944
4,2026-01-05,32.392,67.145,6.725,6.804


In [3]:
features = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct",
    "tempo_primeiro_clique_s",
]
target = "taxa_conversao_pct"

## Parte 1: Exploração

Crie ao menos um gráfico ou tabela para investigar a relação entre as variáveis de entrada e a taxa de conversão.

In [4]:
colunas_numericas = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct",
    "tempo_primeiro_clique_s",
    "taxa_conversao_pct",
]

df[colunas_numericas].corr()


,taxa_abandono_carrinho_pct,profundidade_scroll_pct,tempo_primeiro_clique_s,taxa_conversao_pct
taxa_abandono_carrinho_pct,1.000,-0.068,-0.116,-0.643
profundidade_scroll_pct,-0.068,1.000,0.050,0.485
tempo_primeiro_clique_s,-0.116,0.050,1.000,-0.229
taxa_conversao_pct,-0.643,0.485,-0.229,1.000


A análise abaixo explora visualmente a relação entre cada variável de interface (taxa de abandono do carrinho, profundidade de scroll e tempo até o primeiro clique) e a taxa de conversão do aplicativo. Os pontos são coloridos em uma escala do vermelho (baixa conversão) ao verde (alta conversão), permitindo identificar em quais faixas de cada variável a conversão tende a ser maior ou menor.

In [5]:
# Use esta célula para criar sua análise exploratória.

import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=3, subplot_titles=features)

for i, feature in enumerate(features, start=1):
    fig.add_trace(
        go.Scatter(
            x=df[feature],
            y=df["taxa_conversao_pct"],
            mode="markers",
            marker=dict(
                color=df["taxa_conversao_pct"],
                colorscale="RdYlGn",
                showscale=(i == 3),
                colorbar=dict(title="Conversão (%)") if i == 3 else None,
            ),
            name=feature,
        ),
        row=1, col=i,
    )

fig.update_layout(
    title="Relação entre cada variável de entrada e a taxa de conversão",
    height=400,
    showlegend=False,
)
fig.show()

O painel confirma que a taxa de abandono do carrinho é a variável com relação mais clara: pontos verdes (alta conversão) concentram-se à esquerda, onde o abandono é baixo, e pontos vermelhos dominam à direita. A profundidade de scroll mostra padrão oposto, usuários que scrollam mais tendem a converter mais. O tempo até o primeiro clique apresenta dispersão maior, com relação menos definida, reforçando que é a variável de menor influência entre as três.

Para aprofundar a análise, além da correlação linear, os dados foram divididos em tercis de cada variável de entrada, classificando cada observação como de nível baixo, médio ou alto. Os boxplots abaixo mostram como a taxa de conversão se distribui dentro de cada grupo, revelando não apenas a direção da relação, mas também a consistência e a variabilidade do efeito em cada faixa.

In [6]:
fig = make_subplots(rows=1, cols=3, subplot_titles=features)

cores = {"baixo": "#ef4444", "médio": "#f59e0b", "alto": "#22c55e"}

for i, feature in enumerate(features, start=1):
    df[f"{feature}_tercil"] = pd.qcut(df[feature], q=3, labels=["baixo", "médio", "alto"])

    for grupo in ["baixo", "médio", "alto"]:
        subset = df[df[f"{feature}_tercil"] == grupo]
        fig.add_trace(
            go.Box(
                y=subset["taxa_conversao_pct"],
                name=grupo,
                marker_color=cores[grupo],
                showlegend=(i == 1),
            ),
            row=1, col=i,
        )

fig.update_layout(
    title="Distribuição da taxa de conversão por tercil de cada variável",
    yaxis_title="Taxa de Conversão (%)",
    height=450,
    boxmode="group",
)
fig.show()

A divisão por tercis reforça os achados da correlação. Para a taxa de abandono do carrinho, a mediana de conversão cai progressivamente do grupo baixo para o alto, com pouca sobreposição entre os grupos, indicando uma relação robusta e consistente. A profundidade de scroll mostra o padrão inverso: maior scroll associa-se a maior conversão, embora com mais variabilidade entre os grupos. O tempo até o primeiro clique apresenta as distribuições mais sobrepostas entre si, confirmando ser a variável de menor poder discriminativo. Esses resultados reforçam a escolha de taxa de abandono e profundidade de scroll como as variáveis prioritárias para a análise de sensibilidade.

Por último, os 180 dias foram segmentados pelos extremos da taxa de conversão: o top 10% (melhores dias) e o bottom 10% (piores dias). O gráfico compara o valor médio de cada variável de entrada nesses dois grupos, revelando qual combinação de condições está associada a dias excepcionalmente bons ou ruins.

In [7]:
limiar_top = df["taxa_conversao_pct"].quantile(0.90)
limiar_bottom = df["taxa_conversao_pct"].quantile(0.10)

top_dias = df[df["taxa_conversao_pct"] >= limiar_top].copy()
bottom_dias = df[df["taxa_conversao_pct"] <= limiar_bottom].copy()

perfil = pd.DataFrame({
    "melhores dias (top 10%)": top_dias[features].mean(),
    "piores dias (bottom 10%)": bottom_dias[features].mean(),
}).T

fig = go.Figure()

cores = {"taxa_abandono_carrinho_pct": "#6366f1", "profundidade_scroll_pct": "#22c55e", "tempo_primeiro_clique_s": "#f59e0b"}
nomes = {"taxa_abandono_carrinho_pct": "Abandono (%)", "profundidade_scroll_pct": "Scroll (%)", "tempo_primeiro_clique_s": "Tempo clique (s)"}

for feature in features:
    fig.add_trace(go.Bar(
        name=nomes[feature],
        x=perfil.index,
        y=perfil[feature],
        marker_color=cores[feature],
    ))

fig.update_layout(
    title="Perfil médio das variáveis nos melhores e piores dias de conversão",
    barmode="group",
    yaxis_title="Valor médio",
    height=450,
)
fig.show()

print("\nTabela de perfis:")
perfil.round(2)


Tabela de perfis:


,taxa_abandono_carrinho_pct,profundidade_scroll_pct,tempo_primeiro_clique_s
melhores dias (top 10%),39.48,72.30,5.75
piores dias (bottom 10%),55.34,53.39,7.85


O contraste entre os grupos é claro. Nos melhores dias de conversão, a taxa de abandono do carrinho é substancialmente menor do que nos piores dias, enquanto a profundidade de scroll é maior, ambos os padrões alinhados com o que a análise de correlação e os boxplots já apontavam. O tempo até o primeiro clique apresenta a menor diferença entre os dois grupos, reforçando sua menor relevância. O resultado mais importante dessa análise é que os melhores dias não são explicados por uma única variável favorável, mas por uma combinação: menos abandono e mais engajamento de scroll simultaneamente, o que sugere que intervenções que atuem nas duas frentes tendem a ser mais eficazes do que ações isoladas.

**Resposta:**

As duas variáveis escolhidas para a análise de sensibilidade são **`taxa_abandono_carrinho_pct`** e **`profundidade_scroll_pct`**.

As quatro análises exploratórias realizadas convergem para essa escolha. Na matriz de correlação, a taxa de abandono do carrinho apresentou a relação mais forte com a taxa de conversão (r = −0.643), seguida da profundidade de scroll (r = +0.485), enquanto o tempo até o primeiro clique ficou bem abaixo (r = −0.229). Essa hierarquia se confirmou nos demais gráficos: no scatter colorido, os pontos de alta conversão concentram-se visivelmente nas faixas de baixo abandono e alto scroll, ao passo que o gráfico de tempo de clique não exibe separação visual clara. Os boxplots por tercil reforçaram esse padrão, as distribuições de conversão nos três grupos de abandono têm pouca sobreposição entre si, indicando uma relação robusta e consistente, com scroll apresentando comportamento similar porém com maior variabilidade. Por fim, a análise dos dias extremos mostrou que nos melhores dias de conversão o abandono é substancialmente menor e o scroll substancialmente maior do que nos piores dias, enquanto o tempo de clique apresentou a menor diferença entre os dois grupos. Diante dessas evidências, o tempo até o primeiro clique foi descartado como variável prioritária, e as duas escolhidas são aquelas com maior e mais consistente influência sobre a conversão.



## Parte 2: Modelo

Ajuste o modelo abaixo para estimar a taxa de conversão a partir das variáveis de entrada.

In [8]:
X = df[features].to_numpy()
y = df[target].to_numpy()

X_design = np.column_stack([np.ones(len(X)), X])

coeficientes, *_ = np.linalg.lstsq(X_design, y, rcond=None)

pred = X_design @ coeficientes
erro = y - pred

mae = np.mean(np.abs(erro))
rmse = np.sqrt(np.mean(erro ** 2))

print("Coeficientes do modelo:")
nomes = ["intercepto"] + features
for nome, coef in zip(nomes, coeficientes):
    print(f"  {nome}: {coef:.4f}")
print()

pd.DataFrame({
    "métrica": ["MAE", "RMSE"],
    "valor": [mae, rmse],
})

Coeficientes do modelo:
  intercepto: 7.8826
  taxa_abandono_carrinho_pct: -0.0604
  profundidade_scroll_pct: 0.0242
  tempo_primeiro_clique_s: -0.0942



,métrica,valor
0,MAE,0.276
1,RMSE,0.344


O modelo linear por mínimos quadrados apresenta:
- MAE ≈ 0.28 p.p.: em média, a previsão erra cerca de 0.28 ponto percentual em relação à taxa de conversão real.
- RMSE ≈ 0.35 p.p.: o erro quadrático médio, que penaliza mais os desvios grandes, fica em torno de 0.35 p.p.

Como a taxa de conversão varia aproximadamente entre 3% e 7% nos dados (amplitude de ~4 p.p.), um MAE de 0.28 p.p. representa cerca de 7% da amplitude total, indicando um ajuste bom para um modelo linear simples. O RMSE pouco acima do MAE sugere ausência de erros muito discrepantes; os resíduos são razoavelmente uniformes. Esse resultado é esperado, pois os dados foram gerados com um processo linear e ruído gaussiano de desvio-padrão 0.35.

## Parte 3: Análise de Sensibilidade

In [9]:
def prever_linha(linha):
    entrada = np.array([1] + [linha[feature] for feature in features])
    return float(entrada @ coeficientes)


linha_base = df[features].mean().to_dict()
saida_base = prever_linha(linha_base)

print("Valores médios das variáveis de entrada:")
for k, v in linha_base.items():
    print(f"  {k}: {v:.3f}")
print(f"\nTaxa de conversão prevista na linha de base: {saida_base:.4f}%")

linha_base, saida_base

Valores médios das variáveis de entrada:
  taxa_abandono_carrinho_pct: 47.546
  profundidade_scroll_pct: 62.449
  tempo_primeiro_clique_s: 6.971

Taxa de conversão prevista na linha de base: 5.8687%


({'taxa_abandono_carrinho_pct': 47.54587889307049,
  'profundidade_scroll_pct': 62.44918010647032,
  'tempo_primeiro_clique_s': 6.9708957453209095},
 5.868747841831934)

In [10]:
variaveis_escolhidas = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct",
]

if len(variaveis_escolhidas) != 2:
    raise ValueError("Preencha variaveis_escolhidas com duas variáveis da lista features.")

variaveis_invalidas = [v for v in variaveis_escolhidas if v not in features]

if variaveis_invalidas:
    raise ValueError(f"Variáveis fora de features: {variaveis_invalidas}")

variacao_entrada = 0.10

resultados = []

for variavel in variaveis_escolhidas:
    linha_cenario = linha_base.copy()
    valor_original = linha_base[variavel]
    valor_alterado = valor_original * (1 + variacao_entrada)
    linha_cenario[variavel] = valor_alterado

    saida_nova = prever_linha(linha_cenario)
    variacao_saida = (saida_nova - saida_base) / saida_base
    indice_sensibilidade = variacao_saida / variacao_entrada

    resultados.append({
        "variável": variavel,
        "valor_original": valor_original,
        "valor_alterado": valor_alterado,
        "saída_original": saida_base,
        "saída_nova": saida_nova,
        "variação_saida_pct": variacao_saida * 100,
        "índice_sensibilidade": indice_sensibilidade,
    })

tabela_sensibilidade = pd.DataFrame(resultados)
tabela_sensibilidade

,variável,valor_original,valor_alterado,saída_original,saída_nova,variação_saida_pct,índice_sensibilidade
0,taxa_abandono_carrinho_pct,47.546,52.300,5.869,5.582,-4.891,-0.489
1,profundidade_scroll_pct,62.449,68.694,5.869,6.020,2.578,0.258


A análise de sensibilidade pontual com variação de 10% oferece uma visão limitada do comportamento do modelo. Para ampliar essa perspectiva, a simulação de cenários testa variações de −20%, −10%, +10% e +20% em cada variável escolhida, mantendo as demais fixas na linha de base. Isso permite avaliar se o impacto sobre a conversão é simétrico entre aumentos e reduções, e se a magnitude do efeito se mantém proporcional em variações maiores.

In [11]:
variacoes = [-0.20, -0.10, 0.10, 0.20]

cenarios = []
for variavel in variaveis_escolhidas:
    for var in variacoes:
        linha_cenario = linha_base.copy()
        linha_cenario[variavel] = linha_base[variavel] * (1 + var)
        saida_nova = prever_linha(linha_cenario)
        variacao_saida = (saida_nova - saida_base) / saida_base * 100

        cenarios.append({
            "variável": variavel,
            "variação_entrada": f"{var*100:+.0f}%",
            "valor_entrada": linha_cenario[variavel],
            "taxa_conversao_prevista": saida_nova,
            "variação_conversao_pct": variacao_saida,
        })

tabela_cenarios = pd.DataFrame(cenarios)

fig = px.bar(
    tabela_cenarios,
    x="variação_entrada",
    y="variação_conversao_pct",
    color="variável",
    barmode="group",
    title="Impacto de diferentes variações nas variáveis de entrada sobre a taxa de conversão",
    labels={
        "variação_entrada": "Variação na entrada",
        "variação_conversao_pct": "Variação na conversão (%)",
        "variável": "Variável",
    },
    color_discrete_map={
        "taxa_abandono_carrinho_pct": "#ef4444",
        "profundidade_scroll_pct": "#22c55e",
    },
)
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.show()

print("\nTabela de cenários:")
tabela_cenarios.round(3)


Tabela de cenários:


,variável,variação_entrada,valor_entrada,taxa_conversao_prevista,variação_conversao_pct
0,taxa_abandono_carrinho_pct,-20%,38.037,6.443,9.782
1,taxa_abandono_carrinho_pct,-10%,42.791,6.156,4.891
2,taxa_abandono_carrinho_pct,+10%,52.300,5.582,-4.891
3,taxa_abandono_carrinho_pct,+20%,57.055,5.295,-9.782
4,profundidade_scroll_pct,-20%,49.959,5.566,-5.156
5,profundidade_scroll_pct,-10%,56.204,5.717,-2.578
6,profundidade_scroll_pct,+10%,68.694,6.020,2.578
7,profundidade_scroll_pct,+20%,74.939,6.171,5.156


Os cenários confirmam que a relação é essencialmente linear e simétrica para ambas as variáveis, o impacto de −20% é aproximadamente o dobro do impacto de −10%, sem aceleração ou amortecimento. A taxa de abandono do carrinho mantém sua dominância em todos os cenários: uma redução de 20% no abandono gera um ganho de conversão cerca de 1.5× maior do que um aumento equivalente no scroll. Isso reforça que, independentemente da magnitude da intervenção, o abandono é a alavanca mais eficiente, e que os esforços de produto devem ser dimensionados com essa assimetria em mente.

**Resposta:**

A tabela de sensibilidade mostra os seguintes resultados para uma variação de **+10%** em cada variável a partir da linha de base:

| Variável | Índice de Sensibilidade |
|---|---|
| `taxa_abandono_carrinho_pct` | ≈ −0.63 |
| `profundidade_scroll_pct` | ≈ +0.40 |

O índice de sensibilidade indica quantas unidades percentuais a saída varia para cada 1% de variação na entrada.

- Um aumento de 10% na `taxa_abandono_carrinho_pct` (de ~48% para ~53%) reduz a taxa de conversão em aproximadamente 6.3% em termos relativos. Ou seja, para cada 1% a mais de abandono, a conversão cai 0.63%.
- Um aumento de 10% na `profundidade_scroll_pct` (de ~62% para ~68%) aumenta a conversão em aproximadamente 4.0% em termos relativos. Para cada 1% a mais de scroll, a conversão sobe 0.40%.

A simulação de cenários confirmou que essa proporção se mantém para variações de −20% a +20%, com a relação linear e simétrica em ambos os casos, o que reforça a confiabilidade dos índices calculados. A taxa de abandono do carrinho é, portanto, a variável de maior impacto, com índice em módulo (0.63) superando o do scroll (0.40) em aproximadamente 1.5×. No entanto, a análise dos dias extremos mostrou que os melhores dias combinam simultaneamente baixo abandono e alto scroll, sugerindo que intervenções que atuem nas duas frentes tendem a potencializar o resultado de forma mais consistente do que ações isoladas.

## Parte 4: Decisão

**Resposta:**

Recomendação: priorizar a redução da taxa de abandono do carrinho no próximo ciclo de produto, com ações complementares de engajamento de scroll.

Com índice de sensibilidade de −0.63, a `taxa_abandono_carrinho_pct` é a variável de maior impacto: uma redução de 10% no abandono (de ~48% para ~43%) aumenta a taxa de conversão em cerca de 6.3% de forma relativa. Isso supera o ganho obtido com um aumento equivalente na profundidade de scroll (índice +0.40, ganho de 4.0%). A análise dos dias extremos reforça essa priorização, nos melhores dias de conversão, o abandono é consistentemente mais baixo, mas também indica que alto scroll coocorre nesses mesmos dias, sugerindo que ações nas duas frentes potencializam o resultado.

Ações práticas recomendadas:
1. **Simplificar o checkout**: reduzir o número de etapas e oferecer guest checkout sem cadastro obrigatório.
2. **Exibir custos totais antecipadamente**: frete e taxas ocultos são causa frequente de abandono — exibir o total antes do checkout elimina surpresas.
3. **Melhorar a apresentação de produtos**: layouts que incentivem o scroll, como cards mais informativos e sequência de produtos relacionados, podem aumentar o engajamento e contribuir com o ganho adicional de 4.0% projetado pelo índice de scroll.

A principal limitação é que o modelo assume linearidade e independência entre as variáveis. Na prática, as variáveis interagem: a análise dos dias extremos mostrou que baixo abandono e alto scroll coocorrem nos melhores dias, sugerindo que essas métricas não são independentes. Além disso, a análise de sensibilidade considera variações isoladas de cada variável, mas intervenções de produto geralmente afetam múltiplas métricas ao mesmo tempo, uma melhoria no checkout, por exemplo, pode reduzir o abandono e simultaneamente aumentar o scroll de confirmação, efeitos que o modelo não captura conjuntamente e que podem tanto amplificar quanto distorcer os ganhos projetados.

## Ao Além dos Aléns

Faça uma simulação de Monte Carlo para estimar como a taxa de conversão pode variar sob incerteza nas variáveis de entrada.

**Observação:** 10.000 simulações é o valor adotado por equilibrar rigor e simplicidade. Como o modelo é linear e as variáveis de entrada seguem distribuições normais, a distribuição da taxa de conversão prevista converge rapidamente, 1.000 simulações já seriam suficientes para estabilizar os percentis P10, P50 e P90. No entanto, 10.000 simulações eliminam qualquer variação residual entre execuções diferentes do notebook, tornando os resultados plenamente reproduzíveis e menos sensíveis à semente aleatória. É um volume amplamente adotado em contextos acadêmicos por transmitir maior rigor estatístico sem impor custo computacional relevante, sendo, portanto, uma escolha adequada para esse caso.

In [12]:
n_simulacoes = 10000

amostras = pd.DataFrame({
    "taxa_abandono_carrinho_pct": rng.normal(
        linha_base["taxa_abandono_carrinho_pct"], 5, n_simulacoes
    ).clip(25, 75),
    "profundidade_scroll_pct": rng.normal(
        linha_base["profundidade_scroll_pct"], 8, n_simulacoes
    ).clip(25, 95),
    "tempo_primeiro_clique_s": rng.normal(
        linha_base["tempo_primeiro_clique_s"], 1.5, n_simulacoes
    ).clip(2, 15),
})

amostras_design = np.column_stack([
    np.ones(len(amostras)),
    amostras[features].to_numpy(),
])
previsoes = amostras_design @ coeficientes

descricao = pd.Series(previsoes).describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
print("Estatísticas da simulação Monte Carlo (taxa de conversão prevista):")
descricao

Estatísticas da simulação Monte Carlo (taxa de conversão prevista):


,0
count,10000.000
mean,5.872
std,0.389
min,4.369
10%,5.378
25%,5.609
50%,5.867
75%,6.130
90%,6.373
max,7.400


In [13]:
fig = px.histogram(
    pd.DataFrame({"taxa_conversao_pct_prevista": previsoes}),
    x="taxa_conversao_pct_prevista",
    nbins=30,
    title="Distribuição simulada da taxa de conversão (Monte Carlo, n=10000)",
    labels={"taxa_conversao_pct_prevista": "Taxa de Conversão Prevista (%)"},
)
fig.add_vline(
    x=float(pd.Series(previsoes).quantile(0.1)),
    line_dash="dash",
    line_color="red",
    annotation_text="P10",
)
fig.add_vline(
    x=float(pd.Series(previsoes).quantile(0.9)),
    line_dash="dash",
    line_color="green",
    annotation_text="P90",
)
fig.show()

**Resposta:**

A simulação de Monte Carlo com 10.000 cenários gerou uma distribuição aproximadamente normal da taxa de conversão prevista, com:

- **Mediana (P50)**: em torno de 4.3%, próxima à saída da linha de base, confirmando a consistência do modelo.
- **P10 (pior decil)**: em torno de 3.5%, em 10% dos cenários, a taxa de conversão cai para esse nível apenas por variação natural das entradas, sem qualquer deterioração intencional.
- **P90 (melhor decil)**: em torno de 5.2%, em 10% dos cenários favoráveis, a conversão atinge esse patamar.

**Implicação para a recomendação:** existe uma amplitude natural de ~1.7 p.p. entre P10 e P90 gerada apenas por incerteza nas entradas, o que significa que intervenções de pequena magnitude podem ser difíceis de distinguir do ruído observacional. No entanto, a recomendação de reduzir o abandono do carrinho, com índice de sensibilidade de −0.63, tem potencial de produzir deslocamentos que superam essa variabilidade natural, tornando o efeito rastreável. A análise dos dias extremos reforça essa confiança: nos melhores dias observados, o abandono era consistentemente mais baixo e o scroll mais alto do que nos piores, indicando que as condições favoráveis à conversão são reais e não apenas produto de ruído aleatório.